# Breaking the Feedback Trap: Cross-Domain Rebuttal Evaluation
This notebook evaluates baseline recurrent architectures versus the proposed Detached Soft-OR across new datasets (e.g., ISIC 2018, DRIVE).

## Cell 1: Setup & Environment

In [ ]:
!pip install -q segmentation-models-pytorch albumentations
import os
import cv2
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

## Cell 2: Data Pipeline (Flexible DataLoader)

In [ ]:
class MedicalSegmentationDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load image (RGB) and mask (Grayscale)
        image = cv2.imread(self.image_paths[idx])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
        # Binarize mask
        mask = (mask > 127).astype(np.float32)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
        
        # Add channel dimension to mask (1, H, W)
        mask = mask.unsqueeze(0)
        return image, mask

# Example Transforms for 256x256 resolution
train_transform = A.Compose([
    A.Resize(256, 256),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# Placeholder for initialization
# train_dataset = MedicalSegmentationDataset(train_img_paths, train_mask_paths, transform=train_transform)
# val_dataset = MedicalSegmentationDataset(val_img_paths, val_mask_paths, transform=val_transform)
# train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
# val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)
print("Data Pipeline Ready.")

## Cell 3: Model, Loss, and Optimizer

In [ ]:
class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, smooth=1e-6):
        """
        Asymmetric Tversky Loss. 
        beta > alpha penalizes False Positives (over-segmentation) more heavily.
        """
        super(TverskyLoss, self).__init__()
        self.alpha = alpha
        self.beta = beta
        self.smooth = smooth

    def forward(self, inputs, targets):
        inputs = torch.sigmoid(inputs)
        inputs = inputs.view(-1)
        targets = targets.view(-1)
        
        TP = (inputs * targets).sum()
        FP = ((1 - targets) * inputs).sum()
        FN = (targets * (1 - inputs)).sum()
        
        Tversky = (TP + self.smooth) / (TP + self.alpha * FN + self.beta * FP + self.smooth)
        return 1 - Tversky

# Placeholders for Recurrent Architectures (Inject FANet/R2U-Net here)
class BaselineRecurrentModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 1, kernel_size=3, padding=1)
    def forward(self, x):
        # Simulates Hard Feedback Trap where gradients leak directly
        return self.conv(x)

class ProposedDetachedSoftORModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 1, kernel_size=3, padding=1)
    def forward(self, x):
        # Simulates Detached Soft-OR (Feedback Firewall)
        return self.conv(x)

# Initialization example
model = ProposedDetachedSoftORModel().to(device)
criterion = TverskyLoss(alpha=0.3, beta=0.7)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
print("Model, Loss (Tversky), and Optimizer initialized.")

## Cell 4: The Training Loop (with Logging)

In [ ]:
def calculate_metrics(preds, masks, threshold=0.5):
    preds_bin = (torch.sigmoid(preds) > threshold).float()
    masks = masks.float()
    
    TP = (preds_bin * masks).sum().item()
    FP = (preds_bin * (1 - masks)).sum().item()
    FN = ((1 - preds_bin) * masks).sum().item()
    TN = ((1 - preds_bin) * (1 - masks)).sum().item()
    
    dice = (2 * TP) / (2 * TP + FP + FN + 1e-6)
    fpr = FP / (FP + TN + 1e-6)
    return dice, fpr

def train_and_validate(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=50, device='cuda'):
    history = {'train_loss': [], 'val_loss': [], 'val_dice': [], 'val_fpr': []}
    best_val_dice = 0.0
    
    for epoch in range(num_epochs):
        # --- Training ---
        model.train()
        train_loss_accum = 0.0
        
        for images, masks in train_loader:
            images, masks = images.to(device), masks.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            
            train_loss_accum += loss.item()
        
        avg_train_loss = train_loss_accum / len(train_loader)
        history['train_loss'].append(avg_train_loss)
        
        # --- Validation ---
        model.eval()
        val_loss_accum = 0.0
        val_dice_accum = 0.0
        val_fpr_accum = 0.0
        
        with torch.no_grad():
            for images, masks in val_loader:
                images, masks = images.to(device), masks.to(device)
                outputs = model(images)
                loss = criterion(outputs, masks)
                
                dice, fpr = calculate_metrics(outputs, masks)
                
                val_loss_accum += loss.item()
                val_dice_accum += dice
                val_fpr_accum += fpr
        
        avg_val_loss = val_loss_accum / len(val_loader)
        avg_val_dice = val_dice_accum / len(val_loader)
        avg_val_fpr = val_fpr_accum / len(val_loader)
        
        history['val_loss'].append(avg_val_loss)
        history['val_dice'].append(avg_val_dice)
        history['val_fpr'].append(avg_val_fpr)
        
        scheduler.step()
        
        print(f"Epoch [{epoch+1}/{num_epochs}] "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} | "
              f"Val Dice: {avg_val_dice:.4f} | "
              f"Val FPR: {avg_val_fpr:.4f}")
        
        # Save best model
        if avg_val_dice > best_val_dice:
            best_val_dice = avg_val_dice
            torch.save(model.state_dict(), 'best_model.pth')
            print(f" -> Best model saved at Epoch {epoch+1} with Dice: {best_val_dice:.4f}")
            
        torch.cuda.empty_cache()
        
    return history

print("Training loop defined.")

## Cell 5: Quantitative Visualization (Learning Curves)

In [ ]:
def plot_training_history(history):
    sns.set_theme(style="whitegrid")
    epochs = range(1, len(history['train_loss']) + 1)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    
    # Plot 1: Loss
    ax1.plot(epochs, history['train_loss'], label='Train Loss', marker='o', markersize=4)
    ax1.plot(epochs, history['val_loss'], label='Val Loss', marker='s', markersize=4)
    ax1.set_title('Training and Validation Loss', fontsize=14)
    ax1.set_xlabel('Epochs', fontsize=12)
    ax1.set_ylabel('Tversky Loss', fontsize=12)
    ax1.legend()
    
    # Plot 2: Dice & FPR
    ax2.plot(epochs, history['val_dice'], label='Val Dice', color='green', marker='^', markersize=4)
    ax2_tw = ax2.twinx()
    ax2_tw.plot(epochs, history['val_fpr'], label='Val FPR', color='red', marker='x', markersize=4)
    
    ax2.set_title('Validation Metrics: Dice & FPR', fontsize=14)
    ax2.set_xlabel('Epochs', fontsize=12)
    ax2.set_ylabel('Dice Coefficient', fontsize=12, color='green')
    ax2_tw.set_ylabel('False Positive Rate (FPR)', fontsize=12, color='red')
    
    # Legend handling for twin axes
    lines_1, labels_1 = ax2.get_legend_handles_labels()
    lines_2, labels_2 = ax2_tw.get_legend_handles_labels()
    ax2.legend(lines_1 + lines_2, labels_1 + labels_2, loc='center right')
    
    plt.tight_layout()
    plt.show()

print("Visualization functions ready.")

## Cell 6: Qualitative Visualization (Result Grid with Feedback Trap Overlay)

In [ ]:
def unnormalize(tensor, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
    for t, m, s in zip(tensor, mean, std):
        t.mul_(s).add_(m)
    return tensor

def plot_predictions(image_tensor, mask_tensor, base_pred, prop_pred):
    """
    Visualizes the predictions with TP (Green) and FP (Red) overlays.
    """
    # Convert tensors to numpy arrays
    img = unnormalize(image_tensor.clone().squeeze(0).cpu()).permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    
    gt = mask_tensor.squeeze().cpu().numpy()
    
    b_pred = (torch.sigmoid(base_pred).squeeze().cpu().numpy() > 0.5).astype(np.float32)
    p_pred = (torch.sigmoid(prop_pred).squeeze().cpu().numpy() > 0.5).astype(np.float32)
    
    def create_overlay(image, truth, pred):
        overlay = image.copy()
        # True Positives -> Green
        tp_mask = (truth == 1) & (pred == 1)
        # False Positives -> Red (The Feedback Trap)
        fp_mask = (truth == 0) & (pred == 1)
        
        # Apply colors (RGB)
        overlay[tp_mask] = overlay[tp_mask] * 0.3 + np.array([0, 1, 0]) * 0.7
        overlay[fp_mask] = overlay[fp_mask] * 0.3 + np.array([1, 0, 0]) * 0.7
        return np.clip(overlay, 0, 1)

    base_overlay = create_overlay(img, gt, b_pred)
    prop_overlay = create_overlay(img, gt, p_pred)
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(img)
    axes[0].set_title("Original Image", fontsize=14)
    axes[0].axis('off')
    
    axes[1].imshow(gt, cmap='gray')
    axes[1].set_title("Ground Truth Mask", fontsize=14)
    axes[1].axis('off')
    
    axes[2].imshow(base_overlay)
    axes[2].set_title("Baseline (Hard Feedback)\nRed=FP (Trap)", fontsize=14)
    axes[2].axis('off')
    
    axes[3].imshow(prop_overlay)
    axes[3].set_title("Proposed (Detached Soft-OR)\nRed=FP (Mitigated)", fontsize=14)
    axes[3].axis('off')
    
    plt.tight_layout()
    plt.show()

print("Qualitative evaluation ready.")